# Carga de desembolsos

Proceso de consolidacion de desembolsos del dia.

## 1. Cabecera
> **Descripcion:** Informacion general del proceso: objetivo, version, responsable y tablas involucradas.

In [ ]:
# -------------------------------------------------------------------------
# PROYECTO       : Operaciones - Desembolsos
# PROCESO        : ETL_DESEMBOLSOS
# OBJETIVO       : Consolidar los desembolsos del dia
# VERSION        : 1.0.0
# DESARROLLADOR  : Eduardo Fajardo
# FECHA          : 29/08/2026
# TABLA FUENTE   : mb_silver_prod.ope.h_desembolso
# TABLA DESTINO  : mb_gold_prod.operaciones.fct_desembolso
# FRECUENCIA     : Diaria
# -------------------------------------------------------------------------

## 2. Importacion de librerias
> **Descripcion:** Librerias estandar, de terceros y locales, en ese orden.

In [ ]:
import logging
import time
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 3. Lectura de parametros
> **Descripcion:** Parametros de ejecucion recibidos por widgets, para que el mismo codigo corra en cualquier ambiente.

In [ ]:
dbutils.widgets.text("p_fecha_proceso", "")
dbutils.widgets.text("p_catalogo", "")

var_fecha_proceso = dbutils.widgets.get("p_fecha_proceso")
var_catalogo = dbutils.widgets.get("p_catalogo")

logger = logging.getLogger("ETL_DESEMBOLSOS")
logger.setLevel(logging.INFO)

ini_proceso = time.perf_counter()
logger.info("Inicio del proceso ETL_DESEMBOLSOS")
logger.info("Parametros recibidos: fecha=%s catalogo=%s",
            var_fecha_proceso, var_catalogo)

## 4. Seccion constantes
> **Descripcion:** Valores que se mantienen constantes a lo largo del proceso.

In [ ]:
TBL_DESEMBOLSO_ORIGEN = f"{var_catalogo}.ope.h_desembolso"
TBL_DESEMBOLSO_FINAL = f"{var_catalogo}.operaciones.fct_desembolso"

EST_DESEMBOLSADO = "DESEMBOLSADO"

## 5. Funciones de transformacion
> **Descripcion:** Funciones modularizadas de lectura, transformacion y escritura.

In [ ]:
def read_desembolso(tabla, fecha):
    """Lee los desembolsos del dia proyectando solo las columnas necesarias."""
    return (
        spark.table(tabla)
        .select("cod_operacion", "cod_cliente", "mto_desembolso", "fec_desembolso")
        .filter(F.col("fec_desembolso") == fecha)
    )


def add_moneda_normalizada(df_origen):
    """Normaliza el codigo de moneda a mayusculas."""
    return df_origen.withColumn("tip_moneda", F.upper(F.col("tip_moneda")))

## 6. Logica del proceso
> **Descripcion:** Orquestacion de las funciones definidas previamente.

In [ ]:
ini_etapa = time.perf_counter()

df_desembolso = read_desembolso(TBL_DESEMBOLSO_ORIGEN, var_fecha_proceso)
df_desembolso_final = add_moneda_normalizada(df_desembolso)

logger.info("Tiempo de transformacion: %.2f segundos",
            time.perf_counter() - ini_etapa)

## 7. Deduplicacion
> **Descripcion:** Logica de deduplicacion segun las llaves de la tabla.

In [ ]:
ventana_operacion = Window.partitionBy("cod_operacion").orderBy(
    F.col("fec_desembolso").desc()
)

df_desembolso_unico = (
    df_desembolso_final
    .withColumn("nro_orden", F.row_number().over(ventana_operacion))
    .filter(F.col("nro_orden") == 1)
    .drop("nro_orden")
)

## 8. Escritura en la tabla final
> **Descripcion:** Persistencia del resultado en formato Delta.

In [ ]:
ini_escritura = time.perf_counter()

try:
    (
        df_desembolso_unico
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(TBL_DESEMBOLSO_FINAL)
    )
except Exception as exc:
    logger.error("Fallo la escritura en %s: %s", TBL_DESEMBOLSO_FINAL, exc)
    raise

logger.info("Tiempo de escritura: %.2f segundos",
            time.perf_counter() - ini_escritura)

## 9. Registro de la ejecucion
> **Descripcion:** Cierre del proceso con el registro de duracion y volumen.

In [ ]:
logger.info("Fin del proceso ETL_DESEMBOLSOS. Duracion total: %.2f segundos",
            time.perf_counter() - ini_proceso)